# Philosophy Quote Generator

#### Base:
https://dzone.com/articles/build-a-philosophy-quote-generator-with-vector-sea

https://dzone.com/articles/infinite-wisdom-series-build-a-philosophy-quote-ge

https://dzone.com/articles/build-a-philosophy-quote-generator-with-vector-sea-1

## Initialization

In [29]:
import chromadb
import json
import openai

from openai import OpenAI

from chromadb.utils import embedding_functions
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

In [30]:
import os
from dotenv import load_dotenv

load_dotenv()


True

### Variables

In [15]:
embedding_model_name = "all-mpnet-base-v2" # text-embedding-ada-002
openAI_embedding_model_name = "text-embedding-ada-002"

n_results = 5

user_query = "Solo sé que no sé nada"
query = [user_query]

In [4]:
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=embedding_model_name)
# https://sbert.net/docs/sentence_transformer/pretrained_models.html

C:\Users\Admin\anaconda3\envs\rag-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
openai_ef = OpenAIEmbeddingFunction(
    model_name = openAI_embedding_model_name
)

In [6]:
chroma_client = chromadb.Client()

In [7]:
collection = chroma_client.get_or_create_collection(
    name = "philosphy_quotes2",
    embedding_function = sentence_transformer_ef
)

In [8]:
collectionOpenAI = chroma_client.get_or_create_collection(
    name = "philosophy_quotes_ada",
    embedding_function = openai_ef
)

## Read file

In [9]:
with open('files/quotes.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

    quotes = []
    metadatas = []
    ids = []
    id = 1

    for quote in data["quotes"]:
        quotes.append(quote["quote"])
        metadatas.append({"author": quote["author"]})
        ids.append(str(id))

        id+=1

In [8]:
quotes

['La pluma es la lengua del alma.',
 'El que quiere interesar a los demás tiene que provocarlos.',
 'Todos los niños nacen artistas. El problema es cómo seguir siendo artistas al crecer.',
 'Una es más auténtica, mientras más se parece a lo que soñó de sí misma.',
 'Soy el desesperado, la palabra sin ecos, el que lo perdió todo, y el que todo lo tuvo.',
 'Aprender a sonreír es aprender a ser libres.',
 'Memoria selectiva para recordar lo bueno, prudencia lógica para no arruinar el presente, y optimismo desafiante para encarar el futuro.',
 '¿Se pueden inventar verbos? quiero decirte uno: Yo te cielo, así mis alas se extienden enormes para amarte sin medida.',
 'Se necesitan dos años para aprender a hablar y sesenta para aprender a callar.',
 'No hay que ir para atrás ni para darse impulso.',
 'No hay caminos para la paz; la paz es el camino.',
 'Haz el amor y no la guerra.',
 'Para trabajar basta estar convencido de una cosa: que trabajar es menos aburrido que divertirse.',
 'Lo peor q

In [10]:
collection.add(
    ids = ids,
    metadatas = metadatas,
    documents = quotes
)

In [11]:
collectionOpenAI.add(
    ids = ids,
    metadatas = metadatas,
    documents = quotes
)

## Some trials

In [12]:
quote_results = collection.query(
    query_texts = query,
    n_results = n_results,
    include = ["distances", "metadatas", "documents"]
)

# quote_results

{'ids': [['69', '34', '78', '42', '64']],
 'embeddings': None,
 'documents': [['Solo sé que no sé nada.',
   'El sabio no dice nunca todo lo que piensa, pero siempre piensa todo lo que dice.',
   'El único hombre que no se equivoca es el que nunca hace nada.',
   'Lo que no te mata, te hace más fuerte.',
   'Aquel que más posee, más miedo tiene de perderlo.']],
 'uris': None,
 'included': ['distances', 'metadatas', 'documents'],
 'data': None,
 'metadatas': [[{'author': 'Sócrates'},
   {'author': 'Aristóteles'},
   {'author': 'Goethe'},
   {'author': 'Friedrich Nietzsche'},
   {'author': 'Leonardo Da Vinci'}]],
 'distances': [[0.08885114639997482,
   0.7467907071113586,
   0.7700926065444946,
   0.7865882515907288,
   0.8140975832939148]]}

In [14]:
quote_resultsOpenAI = collectionOpenAI.query(
    query_texts = query,
    n_results = n_results,
    include = ["distances", "metadatas", "documents"]
)

quote_resultsOpenAI

{'ids': [['69', '17', '35', '5', '63']],
 'embeddings': None,
 'documents': [['Solo sé que no sé nada.',
   'Cada día sabemos más y entendemos menos.',
   'Hay dos cosas que son infinitas: el universo y la estupidez humana; de la primera no estoy muy seguro.',
   'Soy el desesperado, la palabra sin ecos, el que lo perdió todo, y el que todo lo tuvo.',
   'Es mejor permanecer callado y parecer tonto que hablar y despejar las dudas definitivamente.']],
 'uris': None,
 'included': ['distances', 'metadatas', 'documents'],
 'data': None,
 'metadatas': [[{'author': 'Sócrates'},
   {'author': 'Albert Einstein'},
   {'author': 'Albert Einstein'},
   {'author': 'Pablo Neruda'},
   {'author': 'Groucho Marx'}]],
 'distances': [[0.04680314660072327,
   0.36304059624671936,
   0.39474213123321533,
   0.3996537923812866,
   0.40254634618759155]]}

In [24]:
client = chromadb.PersistentClient(path="vectordb")

## Quote Generator

In [31]:
prompt = ""
completion_model_name = "gpt-3.5-turbo"
client = OpenAI()

In [34]:
generation_prompt_template = """"Genera una sencilla y corta frase filosófica, dada la siguiente frase de referencia, similar en espíritu, y usando como base los ejemplos
No te excedas de 20 o 30 palabras

FRASE DE REFERENCIA: "{topic}"


EJEMPLOS: "{examples}"

"""


In [21]:
def find_quote_and_author_p(collection, query, n_results, author=None, tags=None):
    quote_results = collection.query(
        query_texts = query,
        n_results = n_results,
        include = ["distances", "metadatas", "documents"]
    )

    return quote_results

In [32]:
def generate_quote(collection, query, n_results=2, author=None, tags=None):
    quotes = find_quote_and_author_p(
        collection = collection,
        query = query,
        n_results = n_results,
        author = author,
        tags = tags
    )

    if quotes:
        prompt = generation_prompt_template.format(
            topic = query[0],
            examples = "\n".join(f"  - {quote[0]}" for quote in quotes),
        )

        response = client.chat.completions.create(
            model=completion_model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
            max_tokens=320
        )

        return response.choices[0].message.content.replace('"', '').strip()

    else:
        print("** no quotes found.")
        return None

In [35]:
generate_quote(collection, query, n_results)

'En la ignorancia encontramos la sabiduría'

In [36]:
generate_quote(collectionOpenAI, query, n_results)

'En la incertidumbre reside la sabiduría'